In [ ]:
import os
import pandas as pd
import numpy as np

from gsm_benchmarker.results_analyser.bootstrap_result import BootstrapResult

from results_notebook_setup import RESULTS_ROOT, RESULTS_FOLDERS


In [ ]:
RESULTS_FOLDERS.keys()

In [ ]:
boot_path = (RESULTS_ROOT / RESULTS_FOLDERS['gsm']).parent / 'bootstrap'
os.listdir(boot_path)

In [ ]:
n_boot = 2000

bs1 = BootstrapResult(boot_path, n_boot=n_boot, glmm_id='variant')

bs2 = BootstrapResult(boot_path, n_boot=n_boot, glmm_id='number')
bs2.summary_df.rename(index={'sum_logs_c': 'gamma_c'}, level=1, inplace=True)
for v in bs2.full_results.values():
    v['estimates'].rename(columns={'sum_logs_c': 'gamma_c'}, inplace=True)


## GLMM 1 - variant effect

In [ ]:
bs1.boot_numbers  # bootstrap stats

In [ ]:
# summary of clean estimates
bs1.boot_df

In [ ]:
# check how many models agree w.r.t. significance of the result; show the ones that don't
bs1.disagreements_check()

In [ ]:
# plot the distribution of estimates in the bootstrap for models which do not agree with single GLMM significance verdict
bs1.plot_nonagreeing_estimates('is_variant')

For all models - also the agreeing ones

In [ ]:
# quick scan for any other model where bootstrap mean diverges meaningfully from the original estimate
# (bias is <bootstrap mean estimate> minus <single GLMM estimate>
bs1.bias_check('is_variant')


In [ ]:
# CI width ratio (bootstrap CI width / original CI width) across all models, excluding the ones which did not converge for single GLMM
bs1.ci_width_check('is_variant')

In [ ]:
# bootstrap estimate distribution skew check across all models
bs1.skew_check('is_variant')

## GLMM2

In [ ]:
bs2.boot_numbers.xs('is_variant', level=1)  # shared for both variables


### a) number effect

In [ ]:
bs2.disagreements_check('gamma_c')

In [ ]:
bs2.plot_nonagreeing_estimates('gamma_c')

In [ ]:
bs2.bias_check('gamma_c')

In [ ]:
bs2.ci_width_check('gamma_c')

In [ ]:
bs2.skew_check('gamma_c')

### b) number-effect-corrected variant effect

In [ ]:
bs2.disagreements_check('is_variant')

In [ ]:
bs2.plot_nonagreeing_estimates('is_variant')

In [ ]:
bs2.bias_check('is_variant')


In [ ]:

bs2.ci_width_check('is_variant')

In [ ]:

bs2.skew_check('is_variant')

---
# Combined analysis for all prompt formats

In [ ]:
n_boot = 2000

boots = {
    'GLMM1': {},
    'GLMM2': {},
}

for prompt_name, prompt_folder in RESULTS_FOLDERS.items():
    boot_path = (RESULTS_ROOT / prompt_folder).parent / 'bootstrap'

    bs1 = BootstrapResult(boot_path, n_boot=n_boot, glmm_id='variant')

    bs2 = BootstrapResult(boot_path, n_boot=n_boot, glmm_id='number')
    bs2.summary_df.rename(index={'sum_logs_c': 'gamma_c'}, level=1, inplace=True)
    for v in bs2.full_results.values():
        v['estimates'].rename(columns={'sum_logs_c': 'gamma_c'}, inplace=True)

    boots['GLMM1'][prompt_name] = bs1
    boots['GLMM2'][prompt_name] = bs2



In [ ]:
bs2.summary_df.xs('is_variant', level=1)

In [ ]:
import pandas as pd

def combine_prompt_boots(glmm_key, effect):
    return pd.concat({key: value.summary_df.xs(effect, level=1) for key, value in boots[glmm_key].items()}, axis=0, names=['prompt', 'model'])

boots_combined = {
    'GLMM1-is_variant': combine_prompt_boots('GLMM1', 'is_variant'),
    'GLMM2-is_variant': combine_prompt_boots('GLMM2', 'is_variant'),
    'GLMM2-gamma_c': combine_prompt_boots('GLMM2', 'gamma_c')
}

boots_combined['GLMM1-is_variant']

In [ ]:
original_significance = boots_combined['GLMM1-is_variant'].xs('gsm', level='prompt').single_significant
significant_models = original_significance[original_significance].index.tolist()
significant_models

In [ ]:
for combo_key, combo_summary in boots_combined.items():
    for prompt_name in combo_summary.index.get_level_values('prompt').unique():
        if 'GLMM1' in combo_key and prompt_name == 'gsm':
            continue
        for model_name in combo_summary.xs(prompt_name, level='prompt').index:
            if model_name not in significant_models:
                combo_summary.drop(index=(prompt_name, model_name), inplace=True)


In [ ]:
combo_summary

In [ ]:
def make_summary(c):
    prompt_group = c.reset_index().groupby('prompt')

    n_models = prompt_group.size()
    n_agreement = prompt_group.agreement.sum()

    convergent_prompt_group = c[~c.single_nonconvergent].reset_index().groupby('prompt')
    width_ratio_mean = convergent_prompt_group.width_ratio.mean()
    width_ratio_sd = convergent_prompt_group.width_ratio.std()

    bias = convergent_prompt_group.bias

    s = pd.concat({
        'N models': n_models,
        'Agreement': pd.Series([f"{a}/{n}" for a, n in zip(n_agreement, n_models)], index=n_agreement.index),
        # 'N singular': prompt_group.single_singular.sum(),
        'N non-convergent': prompt_group.single_nonconvergent.sum(),
        'CI width ratio: mean ± SD': pd.Series([f"{mean:.3f} ± {sd:.3f}" for mean, sd in zip(width_ratio_mean, width_ratio_sd)], index=width_ratio_mean.index),
        'Max|bias|': pd.Series(np.maximum(bias.max(), -bias.min()), index=width_ratio_mean.index),
    }, axis=1).loc[RESULTS_FOLDERS.keys()]
    return s

make_summary(boots_combined['GLMM1-is_variant'])

In [ ]:
make_summary(boots_combined['GLMM2-is_variant'])

In [ ]:
make_summary(boots_combined['GLMM2-gamma_c'])

In [ ]:
boots_combined_single_df = pd.concat(boots_combined, names=['test'])
boots_combined_single_df

In [ ]:
boots_combined_single_df[~boots_combined_single_df.agreement][['single_significant', 'boot_significant', 'single_nonconvergent']].sort_values(['single_nonconvergent']).sort_index(level='prompt')

In [ ]:
boots_combined_single_df[boots_combined_single_df.single_nonconvergent][['agreement', 'single_significant', 'boot_significant']]

In [55]:
bs1.summary_df

,,boot_ci_upper,boot_ci_lower,boot_se,boot_mean,boot_n_requested,boot_n_failed,boot_n_nonconverged,boot_n_singular,boot_n_clean,boot_elapsed_seconds,...,single_singular,single_nonconvergent,single_fit_failed,single_significant,single_ci_width,boot_significant,bias,boot_ci_width,width_ratio,agreement
Mathstral-7B-v0.1,is_variant,0.949648,-0.536633,0.381554,0.229443,2000.0,0.0,120.0,0.0,1880.0,6917.372284,...,False,False,False,False,1.559306,False,-0.020877,1.486281,0.953168,True
Meta-Llama-3-8B,is_variant,0.754696,-0.566404,0.336167,0.094260,2000.0,0.0,0.0,0.0,2000.0,5161.770816,...,False,False,False,False,1.232132,False,0.002923,1.321100,1.072207,True
Meta-Llama-3-8B-Instruct,is_variant,0.650081,-0.819886,0.383100,-0.057620,2000.0,0.0,6.0,0.0,1994.0,5397.139336,...,False,False,False,False,1.290125,False,-0.015356,1.469967,1.139398,True
Mistral-7B-Instruct-v0.1,is_variant,0.369509,-0.896134,0.321093,-0.273058,2000.0,0.0,0.0,0.0,2000.0,4913.246247,...,False,False,False,False,1.171429,False,-0.005896,1.265643,1.080426,True
Phi-3.5-mini-instruct,is_variant,0.393717,-1.411231,0.458298,-0.493948,2000.0,0.0,137.0,0.0,1863.0,7820.852323,...,False,False,False,False,1.587520,False,-0.035648,1.804948,1.136960,True
gemma-2-2b,is_variant,0.359700,-0.978144,0.350209,-0.291799,2000.0,0.0,759.0,0.0,1241.0,5505.280632,...,False,True,False,True,0.001706,False,0.001415,1.337843,784.125380,False
gemma-2-9b,is_variant,0.936238,-1.036487,0.511807,-0.013433,2000.0,0.0,13.0,0.0,1987.0,5845.662810,...,False,False,False,False,1.414825,False,-0.002479,1.972724,1.394324,True
gemma-2b,is_variant,0.433258,-0.993799,0.366529,-0.335530,2000.0,0.0,128.0,0.0,1872.0,5122.778200,...,False,False,False,False,1.429636,False,0.026705,1.427057,0.998196,True
gemma-7b-it,is_variant,0.529063,-1.101247,0.420720,-0.268255,2000.0,0.0,156.0,0.0,1844.0,5645.984661,...,False,False,False,False,1.267248,False,-0.005246,1.630310,1.286497,True
phi-2,is_variant,0.450116,-0.961068,0.358407,-0.264729,2000.0,0.0,20.0,0.0,1980.0,5727.546318,...,False,False,False,False,1.344785,False,0.005764,1.411185,1.049376,True


In [58]:
bs1.summary_df

,,boot_ci_upper,boot_ci_lower,boot_se,boot_mean,boot_n_requested,boot_n_failed,boot_n_nonconverged,boot_n_singular,boot_n_clean,boot_elapsed_seconds,...,single_singular,single_nonconvergent,single_fit_failed,single_significant,single_ci_width,boot_significant,bias,boot_ci_width,width_ratio,agreement
Mathstral-7B-v0.1,is_variant,0.949648,-0.536633,0.381554,0.229443,2000.0,0.0,120.0,0.0,1880.0,6917.372284,...,False,False,False,False,1.559306,False,-0.020877,1.486281,0.953168,True
Meta-Llama-3-8B,is_variant,0.754696,-0.566404,0.336167,0.094260,2000.0,0.0,0.0,0.0,2000.0,5161.770816,...,False,False,False,False,1.232132,False,0.002923,1.321100,1.072207,True
Meta-Llama-3-8B-Instruct,is_variant,0.650081,-0.819886,0.383100,-0.057620,2000.0,0.0,6.0,0.0,1994.0,5397.139336,...,False,False,False,False,1.290125,False,-0.015356,1.469967,1.139398,True
Mistral-7B-Instruct-v0.1,is_variant,0.369509,-0.896134,0.321093,-0.273058,2000.0,0.0,0.0,0.0,2000.0,4913.246247,...,False,False,False,False,1.171429,False,-0.005896,1.265643,1.080426,True
Phi-3.5-mini-instruct,is_variant,0.393717,-1.411231,0.458298,-0.493948,2000.0,0.0,137.0,0.0,1863.0,7820.852323,...,False,False,False,False,1.587520,False,-0.035648,1.804948,1.136960,True
gemma-2-2b,is_variant,0.359700,-0.978144,0.350209,-0.291799,2000.0,0.0,759.0,0.0,1241.0,5505.280632,...,False,True,False,True,0.001706,False,0.001415,1.337843,784.125380,False
gemma-2-9b,is_variant,0.936238,-1.036487,0.511807,-0.013433,2000.0,0.0,13.0,0.0,1987.0,5845.662810,...,False,False,False,False,1.414825,False,-0.002479,1.972724,1.394324,True
gemma-2b,is_variant,0.433258,-0.993799,0.366529,-0.335530,2000.0,0.0,128.0,0.0,1872.0,5122.778200,...,False,False,False,False,1.429636,False,0.026705,1.427057,0.998196,True
gemma-7b-it,is_variant,0.529063,-1.101247,0.420720,-0.268255,2000.0,0.0,156.0,0.0,1844.0,5645.984661,...,False,False,False,False,1.267248,False,-0.005246,1.630310,1.286497,True
phi-2,is_variant,0.450116,-0.961068,0.358407,-0.264729,2000.0,0.0,20.0,0.0,1980.0,5727.546318,...,False,False,False,False,1.344785,False,0.005764,1.411185,1.049376,True
